In [1]:
import os
import json
import pickle
import random
import gc

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from transformers import AutoTokenizer

In [2]:
SEED = 42

SEQUENCE_LENGTH = 100

TRAIN_RATIO = 0.85

FINBERT_MODEL_NAME = "ProsusAI/finbert"
MAX_LENGTH = 64

BATCH_SIZE = 32
TEST_BATCH_SIZE = 64

In [3]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Environment ready.")

Environment ready.


In [4]:
PROJECT_ROOT = os.path.abspath("..")

FI2010_DIR = os.path.join(
    PROJECT_ROOT,
    "datasets",
    "FI2010"
)

FI2010_OUTPUT_DIR = os.path.join(
    PROJECT_ROOT,
    "datasets",
    "processed",
    "fi2010"
)

os.makedirs(
    FI2010_OUTPUT_DIR,
    exist_ok=True
)

TRAIN_PATH = os.path.join(
    FI2010_DIR,
    "train_clean.csv"
)

TEST_PATH = os.path.join(
    FI2010_DIR,
    "test_clean.csv"
)

print(TRAIN_PATH)
print(TEST_PATH)

c:\Users\DELL\Downloads\QuantFormor\QuantFormer-Multimodal-Order-Book-Sentiment-Transformer\datasets\FI2010\train_clean.csv
c:\Users\DELL\Downloads\QuantFormor\QuantFormer-Multimodal-Order-Book-Sentiment-Transformer\datasets\FI2010\test_clean.csv


In [5]:
fi2010_train_df = pd.read_csv(
    TRAIN_PATH
)

fi2010_test_df = pd.read_csv(
    TEST_PATH
)

print(
    "Training shape:",
    fi2010_train_df.shape
)

print(
    "Testing shape:",
    fi2010_test_df.shape
)

Training shape: (31937, 149)
Testing shape: (362400, 149)


In [6]:
CONSTANT_FEATURE = "143"

fi2010_train_df = fi2010_train_df.drop(
    columns=[CONSTANT_FEATURE]
)

fi2010_test_df = fi2010_test_df.drop(
    columns=[CONSTANT_FEATURE]
)

print("Constant feature removed.")

Constant feature removed.


In [7]:
TARGET_COLUMNS = [
    "144",
    "145",
    "146",
    "147",
    "148"
]

TARGET_COLUMN = "148"

FEATURE_COLUMNS = [
    column
    for column in fi2010_train_df.columns
    if column not in TARGET_COLUMNS
]

print(
    "Number of features:",
    len(FEATURE_COLUMNS)
)

Number of features: 143


In [8]:
X_train_full = fi2010_train_df[
    FEATURE_COLUMNS
].copy()

y_train_full = fi2010_train_df[
    TARGET_COLUMN
].copy()

X_test = fi2010_test_df[
    FEATURE_COLUMNS
].copy()

y_test = fi2010_test_df[
    TARGET_COLUMN
].copy()

print("X_train:", X_train_full.shape)
print("y_train:", y_train_full.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (31937, 143)
y_train: (31937,)
X_test: (362400, 143)
y_test: (362400,)


In [9]:
for target in TARGET_COLUMNS:
    assert target not in FEATURE_COLUMNS

print("Target leakage check passed.")

Target leakage check passed.


In [10]:
train_end = int(
    len(X_train_full) * TRAIN_RATIO
)

X_train = X_train_full.iloc[
    :train_end
].copy()

X_val = X_train_full.iloc[
    train_end:
].copy()

y_train = y_train_full.iloc[
    :train_end
].copy()

y_val = y_train_full.iloc[
    train_end:
].copy()

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (27146, 143)
Validation: (4791, 143)
Test: (362400, 143)


In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_val_scaled = scaler.transform(
    X_val
)

X_test_scaled = scaler.transform(
    X_test
)

print("Scaling completed.")

Scaling completed.


In [12]:
LABEL_MAPPING = {
    1: 0,
    2: 1,
    3: 2
}

y_train_encoded = y_train.map(
    LABEL_MAPPING
).to_numpy(
    dtype=np.int64
)

y_val_encoded = y_val.map(
    LABEL_MAPPING
).to_numpy(
    dtype=np.int64
)

y_test_encoded = y_test.map(
    LABEL_MAPPING
).to_numpy(
    dtype=np.int64
)

print(
    "Labels:",
    np.unique(y_train_encoded)
)

Labels: [0 1 2]


In [13]:
X_train_scaled = np.asarray(
    X_train_scaled,
    dtype=np.float32
)

X_val_scaled = np.asarray(
    X_val_scaled,
    dtype=np.float32
)

X_test_scaled = np.asarray(
    X_test_scaled,
    dtype=np.float32
)

print(X_train_scaled.dtype)
print(X_val_scaled.dtype)
print(X_test_scaled.dtype)

float32
float32
float32


In [14]:
def create_sequences(X, y, sequence_length):

    X_sequences = []
    y_sequences = []

    for i in range(
        sequence_length,
        len(X)
    ):

        X_sequences.append(
            X[
                i - sequence_length:i
            ]
        )

        y_sequences.append(
            y[i]
        )

    return (
        np.asarray(
            X_sequences,
            dtype=np.float32
        ),
        np.asarray(
            y_sequences,
            dtype=np.int64
        )
    )

In [15]:
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_encoded,
    SEQUENCE_LENGTH
)

print(
    "X_train:",
    X_train_seq.shape
)

print(
    "y_train:",
    y_train_seq.shape
)

X_train: (27046, 100, 143)
y_train: (27046,)


In [16]:
X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_encoded,
    SEQUENCE_LENGTH
)

print(
    "X_val:",
    X_val_seq.shape
)

print(
    "y_val:",
    y_val_seq.shape
)

X_val: (4691, 100, 143)
y_val: (4691,)


In [17]:
class FI2010TestDataset(Dataset):

    def __init__(
        self,
        X,
        y,
        sequence_length
    ):
        self.X = X
        self.y = y
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.X) - self.sequence_length

    def __getitem__(self, idx):

        X_sequence = self.X[
            idx:idx + self.sequence_length
        ]

        target = self.y[
            idx + self.sequence_length
        ]

        return (
            torch.tensor(
                X_sequence,
                dtype=torch.float32
            ),
            torch.tensor(
                target,
                dtype=torch.long
            )
        )

In [18]:
test_dataset = FI2010TestDataset(
    X_test_scaled,
    y_test_encoded,
    SEQUENCE_LENGTH
)

print(
    "Test sequences:",
    len(test_dataset)
)

Test sequences: 362300


In [19]:
test_loader = DataLoader(
    test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Test DataLoader ready.")

Test DataLoader ready.


In [20]:
test_X, test_y = next(
    iter(test_loader)
)

print(
    "Test X:",
    test_X.shape
)

print(
    "Test y:",
    test_y.shape
)

Test X: torch.Size([64, 100, 143])
Test y: torch.Size([64])


In [21]:
np.save(
    os.path.join(
        FI2010_OUTPUT_DIR,
        "X_train.npy"
    ),
    X_train_seq
)

np.save(
    os.path.join(
        FI2010_OUTPUT_DIR,
        "y_train.npy"
    ),
    y_train_seq
)

np.save(
    os.path.join(
        FI2010_OUTPUT_DIR,
        "X_val.npy"
    ),
    X_val_seq
)

np.save(
    os.path.join(
        FI2010_OUTPUT_DIR,
        "y_val.npy"
    ),
    y_val_seq
)

np.save(
    os.path.join(
        FI2010_OUTPUT_DIR,
        "X_test_scaled.npy"
    ),
    X_test_scaled
)

np.save(
    os.path.join(
        FI2010_OUTPUT_DIR,
        "y_test.npy"
    ),
    y_test_encoded
)

print("FI-2010 preprocessing saved.")

FI-2010 preprocessing saved.


In [22]:
with open(
    os.path.join(
        FI2010_OUTPUT_DIR,
        "standard_scaler.pkl"
    ),
    "wb"
) as f:

    pickle.dump(
        scaler,
        f
    )

print("Scaler saved.")

Scaler saved.


In [23]:
preprocessing_config = {
    "target_column": "148",
    "target_mapping": {
        "1": 0,
        "2": 1,
        "3": 2
    },
    "constant_feature_removed": "143",
    "num_features": 143,
    "sequence_length": 100,
    "train_ratio": 0.85,
    "scaler": "StandardScaler"
}

with open(
    os.path.join(
        FI2010_OUTPUT_DIR,
        "preprocessing_config.json"
    ),
    "w"
) as f:

    json.dump(
        preprocessing_config,
        f,
        indent=4
    )

print("Configuration saved.")

Configuration saved.


In [24]:
PHRASEBANK_DIR = os.path.join(
    PROJECT_ROOT,
    "datasets",
    "FinancialPhraseBank-v1.0"
)

PHRASEBANK_PATH = os.path.join(
    PHRASEBANK_DIR,
    "Sentences_AllAgree.txt"
)

PHRASEBANK_OUTPUT_DIR = os.path.join(
    PROJECT_ROOT,
    "datasets",
    "processed",
    "phrasebank"
)

os.makedirs(
    PHRASEBANK_OUTPUT_DIR,
    exist_ok=True
)

In [25]:
phrasebank_df = pd.read_csv(
    PHRASEBANK_PATH,
    sep="@",
    header=None,
    names=[
        "sentence",
        "sentiment"
    ],
    encoding="latin-1"
)

print(
    "PhraseBank shape:",
    phrasebank_df.shape
)

display(
    phrasebank_df.head()
)

PhraseBank shape: (2264, 2)


,sentence,sentiment
0,"According to Gran , the company has no plans t...",neutral
1,"For the last quarter of 2010 , Componenta 's n...",positive
2,"In the third quarter of 2010 , net sales incre...",positive
3,Operating profit rose to EUR 13.1 mn from EUR ...,positive
4,"Operating profit totalled EUR 21.1 mn , up fro...",positive


In [26]:
phrasebank_df["sentence"] = (
    phrasebank_df["sentence"]
    .astype(str)
    .str.strip()
)

phrasebank_df["sentiment"] = (
    phrasebank_df["sentiment"]
    .astype(str)
    .str.strip()
    .str.lower()
)

print(
    phrasebank_df["sentiment"]
    .value_counts()
)

sentiment
neutral     1391
positive     570
negative     303
Name: count, dtype: int64


In [27]:
SENTIMENT_MAPPING = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

phrasebank_df["label"] = (
    phrasebank_df["sentiment"]
    .map(SENTIMENT_MAPPING)
)

assert phrasebank_df["label"].isna().sum() == 0

print(
    phrasebank_df["label"]
    .value_counts()
    .sort_index()
)

label
0     303
1    1391
2     570
Name: count, dtype: int64


In [28]:
train_df, temp_df = train_test_split(
    phrasebank_df,
    test_size=0.30,
    random_state=SEED,
    stratify=phrasebank_df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (1584, 3)
Validation: (340, 3)
Test: (340, 3)


In [29]:
tokenizer = AutoTokenizer.from_pretrained(
    FINBERT_MODEL_NAME
)

print("FinBERT tokenizer loaded.")

FinBERT tokenizer loaded.


In [30]:
def tokenize_texts(texts):

    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )


train_encodings = tokenize_texts(
    train_df["sentence"].tolist()
)

val_encodings = tokenize_texts(
    val_df["sentence"].tolist()
)

test_encodings = tokenize_texts(
    test_df["sentence"].tolist()
)

print(
    "Train input:",
    train_encodings["input_ids"].shape
)

print(
    "Train mask:",
    train_encodings["attention_mask"].shape
)

Train input: torch.Size([1584, 64])
Train mask: torch.Size([1584, 64])


In [31]:
class FinancialPhraseBankDataset(Dataset):

    def __init__(
        self,
        encodings,
        labels
    ):

        self.encodings = encodings

        self.labels = torch.tensor(
            labels,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            key: value[idx]
            for key, value
            in self.encodings.items()
        }

        item["labels"] = self.labels[idx]

        return item

In [32]:
train_dataset = FinancialPhraseBankDataset(
    train_encodings,
    train_df["label"].tolist()
)

val_dataset = FinancialPhraseBankDataset(
    val_encodings,
    val_df["label"].tolist()
)

test_dataset_phrasebank = FinancialPhraseBankDataset(
    test_encodings,
    test_df["label"].tolist()
)

print(
    len(train_dataset),
    len(val_dataset),
    len(test_dataset_phrasebank)
)

1584 340 340


In [33]:
train_loader_phrasebank = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader_phrasebank = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader_phrasebank = DataLoader(
    test_dataset_phrasebank,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("PhraseBank DataLoaders ready.")

PhraseBank DataLoaders ready.


In [34]:
batch = next(
    iter(train_loader_phrasebank)
)

for key, value in batch.items():

    print(
        key,
        "->",
        value.shape
    )

input_ids -> torch.Size([32, 64])
token_type_ids -> torch.Size([32, 64])
attention_mask -> torch.Size([32, 64])
labels -> torch.Size([32])


In [35]:
torch.save(
    {
        "input_ids": train_encodings["input_ids"],
        "attention_mask": train_encodings["attention_mask"],
        "labels": torch.tensor(
            train_df["label"].tolist(),
            dtype=torch.long
        )
    },
    os.path.join(
        PHRASEBANK_OUTPUT_DIR,
        "train.pt"
    )
)

torch.save(
    {
        "input_ids": val_encodings["input_ids"],
        "attention_mask": val_encodings["attention_mask"],
        "labels": torch.tensor(
            val_df["label"].tolist(),
            dtype=torch.long
        )
    },
    os.path.join(
        PHRASEBANK_OUTPUT_DIR,
        "val.pt"
    )
)

torch.save(
    {
        "input_ids": test_encodings["input_ids"],
        "attention_mask": test_encodings["attention_mask"],
        "labels": torch.tensor(
            test_df["label"].tolist(),
            dtype=torch.long
        )
    },
    os.path.join(
        PHRASEBANK_OUTPUT_DIR,
        "test.pt"
    )
)

print("PhraseBank tensors saved.")

PhraseBank tensors saved.


In [36]:
TOKENIZER_DIR = os.path.join(
    PHRASEBANK_OUTPUT_DIR,
    "finbert_tokenizer"
)

tokenizer.save_pretrained(
    TOKENIZER_DIR
)

with open(
    os.path.join(
        PHRASEBANK_OUTPUT_DIR,
        "label_mapping.json"
    ),
    "w"
) as f:

    json.dump(
        SENTIMENT_MAPPING,
        f,
        indent=4
    )

print("Tokenizer and label mapping saved.")

Tokenizer and label mapping saved.
